# Phase 6: Business and Economic Reward Layer

This notebook runs Phase 6 calibration and inspects the generated artifacts in `Simulation_4/artifacts/phase6/`.

In [ ]:
from pathlib import Path
import json
import subprocess

repo_root = Path.cwd()
if not (repo_root / "Simulation_4").exists():
    repo_root = repo_root.parent

script_path = repo_root / "Simulation_4" / "scripts" / "phase6_reward_model.py"
result = subprocess.run(["python3", str(script_path)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Phase 6 script failed with code {result.returncode}")

In [ ]:
phase6_dir = repo_root / "Simulation_4" / "artifacts" / "phase6"
tier_config = json.loads((phase6_dir / "tier_config.json").read_text())
churn_validation = json.loads((phase6_dir / "churn_validation.json").read_text())
reward_model = json.loads((phase6_dir / "reward_model.json").read_text())

print("Business tier probabilities:")
for k, v in tier_config["business_tier_probabilities"].items():
    print(f"  {k}: {v:.4f}")

print("\nSelected churn coefficients:")
for k, v in churn_validation["coefficients"].items():
    print(f"  {k}: {v:.4f}")

cal = reward_model["calibration"]
print("\nReward calibration:")
print(f"  omega: {cal['omega_selected']:.6f}")
print(f"  p_churn_moderate: {cal['p_churn_moderate']:.4f}")
print(f"  max_churn_cost: {cal['max_churn_cost']:.4f}")

In [ ]:
print("Trajectory totals:")
for name, row in reward_model["trajectory_sanity"].items():
    ok = row["within_target_range_-5_to_5"]
    print(f"  {name}: total={row['episode_total']:.4f}, within_range={ok}")

if reward_model.get("warnings"):
    print("\nWarnings:")
    for w in reward_model["warnings"]:
        print(f"  - {w}")
else:
    print("\nWarnings: none")